#### 1. Loading the data
The extracted scenario events from the five CISA manuals are loaded from a CSV file.

In [30]:
import pandas as pd
from pathlib import Path

data_dir = Path(r"C:\Users\liisiparve\Projects\cybersecurity-project\data\processed")

df_injects = pd.read_csv(data_dir / "cisa_scenario_injects.csv")

print("Events:", df_injects.shape)

Events: (41, 6)


#### 2. Inspecting the events
The scenario descriptions are displayed to check that the extracted information looks correct.

In [26]:
pd.set_option("display.max_colwidth", None)

display(df_injects[
    ["inject_id", "manual_title", "section", "day", "event_description"]
])

,inject_id,manual_title,section,day,event_description
0,HC-INJ-87,Healthcare,Module 1,Day 1,"The Cybersecurity and Infrastructure Security Agency (CISA), the Federal Bureau of Investigation (FBI), and the Department of Health and Human Services (HHS) Health Sector Cybersecurity Coordination Center (HC3) release a joint alert regarding a rise in cyberattacks targeting healthcare organizations. The alert describes the tactics, techniques, and procedures (TTPs) used by cyber criminals, including phishing emails, ransomware, remote hacking, distributed denial of service (DDoS) attacks, and data exfiltration from healthcare organizations."
1,HC-INJ-89,Healthcare,Module 1,Day 9,"<Organization> employees receive e-mail notifications from your managed service provider (MSP) requesting they update their profiles prior to the launch of a new human resources (HR) system. The e-mail contains a link and attachment named Update_Form.docx. Employees working at remote workstations across facilities click the link and are asked to log into their web portal. Upon logging in, they are brought to a 404 Error Page. Some employees also open the attachment and click to allow macros to run."
2,HC-INJ-117,Healthcare,Module 1,Day 10,Manufacturer’s updates are pushed for installation on patient monitors. The updates are downloaded from the manufacturer’s website and installed without issue.
3,HC-INJ-119,Healthcare,Module 1,Day 13,"A third-party electronic medical record (EMR) vendor shows up unannounced at your facility to update equipment. The vendor needs to patch a recently discovered vulnerability in software used on several devices, including workstations, imaging and radiology equipment, bedside monitors, and other clinical devices."
4,HC-INJ-126,Healthcare,Module 1,Day 15,"Radiology reports an issue with their Magnetic Resonance Imaging (MRI) machine accepting patient information, stating their computer is locking up or taking a long time to load. The reports also indicate an extraordinary amount of time is required to pull images for each patient. The delays have resulted in patient appointments running more than two hours behind schedule."
5,HC-INJ-135,Healthcare,Module 2,Day 20,A system administrator discovers active accounts for IT employees that left the organization over the past 12 months. The administrator closes them out.
6,HC-INJ-137,Healthcare,Module 2,Day 23 - Morning,"Nurses on the floor report that patient records are displaying incorrect information about medication, diagnosis, and personal information."
7,HC-INJ-139,Healthcare,Module 2,Day 23 - Afternoon,Staff discover bedside monitor data is inaccurate and the infusion pumps are not operating properly and are failing to deliver infusions at the correct rate.
8,HC-INJ-156,Healthcare,Module 2,Day 25,"Ransomware messages appear on computers throughout <organization>, and users report they are unable to access their files. A message is displayed that reads: “Hello! Your files have been liberated. We have your data. But do not fear because for the sum of $1,000,000 your files will be returned. The decryption key will expire in 72 hours. Please submit payment to the wallet below or we will start selling patient data to the highest bidder.”"
9,HC-INJ-159,Healthcare,Module 2,Day 26,Current and former patients contact the hospital saying they received calls from someone claiming to have access to their medical records and offering to return them for a fee. The patients are provided enough information to verify the callers have their records. Patients say the fees range from a few hundred dollars to more than a thousand and are demanding to know why these individuals have their records. Some say they contacted law enforcement; others contacted the media. Many are threatening legal action.


#### 3. Checking the data
The dataset is checked for missing values, duplicates, and short descriptions. Event counts are also compared across the manuals.

In [27]:
print("Missing values:")
print(df_injects.isna().sum())

print("\nDuplicate IDs:", df_injects["inject_id"].duplicated().sum())
print("Duplicate descriptions:", df_injects["event_description"].duplicated().sum())

short_events = df_injects[df_injects["event_description"].str.len() < 100]
print("\nDescriptions under 100 characters:", len(short_events))

print("\nEvents per manual:")
print(df_injects["manual_title"].value_counts())

print("\nEvents per section:")
display(
    df_injects.groupby(["manual_title", "section"])
    .size()
    .reset_index(name="event_count")
)

Missing values:
manual_title         0
inject_id            0
section              0
day                  2
event_description    0
source_paragraph     0
dtype: int64

Duplicate IDs: 0
Duplicate descriptions: 0

Descriptions under 100 characters: 0

Events per manual:
manual_title
Insider Threat    12
Healthcare        11
Vendor             8
Ransomware         8
Executives         2
Name: count, dtype: int64

Events per section:


,manual_title,section,event_count
0,Executives,Vignette 1: Whaling/Ransomware,1
1,Executives,Vignette 2: Insider Threat,1
2,Healthcare,Module 1,5
3,Healthcare,Module 2,6
4,Insider Threat,Module 1,5
5,Insider Threat,Module 2,7
6,Ransomware,Module 1,4
7,Ransomware,Module 2,4
8,Vendor,Module 1,4
9,Vendor,Module 2 – 6 months later,4


#### 4. Classifying the events
Each scenario event is assigned a threat type and a description of its business consequences based on the CISA manuals.

In [28]:
df_labeled = df_injects.copy()

annotations = {
    # HEALTHCARE: 11 events
    "HC-INJ-87": (
        "Security Alert",
        "No direct impact reported; increased awareness of threats targeting healthcare organizations"
    ),
    "HC-INJ-89": (
        "Suspected Phishing",
        "Potential credential theft and malware infection through suspicious emails and attachments"
    ),
    "HC-INJ-117": (
        "No Specific Threat",
        "No negative impact reported; patient monitor updates installed successfully"
    ),
    "HC-INJ-119": (
        "Vulnerability Exposure",
        "Potential compromise of clinical devices and systems if the vulnerability remains unpatched"
    ),
    "HC-INJ-126": (
        "System Disruption (Unconfirmed Cause)",
        "MRI system delays causing patient appointments to run more than two hours behind schedule"
    ),
    "HC-INJ-135": (
        "Unauthorized Access Risk",
        "Potential misuse of former employees' active IT accounts; accounts were deactivated"
    ),
    "HC-INJ-137": (
        "Data Integrity Issue",
        "Incorrect patient records creating risks to patient safety and clinical decisions"
    ),
    "HC-INJ-139": (
        "Medical Device Malfunction",
        "Inaccurate monitoring data and incorrect infusion rates threatening patient safety"
    ),
    "HC-INJ-156": (
        "Ransomware",
        "Files unavailable, $1 million ransom demand, and threatened exposure of patient data"
    ),
    "HC-INJ-159": (
        "Data Breach",
        "Patient information exposed; extortion attempts, complaints, and threatened legal action"
    ),
    "HC-INJ-163": (
        "Data Breach",
        "Loss of patient trust, requests for hospital transfers, and potential loss of patients"
    ),

    # VENDOR SUPPLY CHAIN: 8 events
    "VD-INJ-111": (
        "Supply Chain Compromise",
        "Global service disruptions affecting MSP customers; no direct impact on the organization"
    ),
    "VD-INJ-122": (
        "No Specific Threat",
        "No negative impact reported during onboarding of a new managed service provider"
    ),
    "VD-INJ-136": (
        "Third-Party Data Breach",
        "Partner data exposed; possible concerns about the security of the new MSP"
    ),
    "VD-INJ-143": (
        "No Specific Threat",
        "No negative impact reported; authorized software update intended to improve functionality"
    ),
    "VD-INJ-150": (
        "System Disruption (Unconfirmed Cause)",
        "Employees unable to access the network; MSP support unavailable"
    ),
    "VD-INJ-152": (
        "System Disruption (Unconfirmed Cause)",
        "Continued network outage significantly affecting business-critical systems"
    ),
    "VD-INJ-162": (
        "Supply Chain Compromise",
        "MSP platform compromised; potential malware exposure affecting customers"
    ),
    "VD-INJ-167": (
        "Suspicious Network Activity",
        "Possible network compromise requiring investigation; full impact not yet established"
    ),

    # RANSOMWARE: 8 events
    "RW-INJ-84": (
        "Security Alert",
        "No direct impact reported; warning about a new ransomware campaign"
    ),
    "RW-INJ-86": (
        "Vulnerability Exposure",
        "Increased security risk because the operating system no longer receives security patches"
    ),
    "RW-INJ-88": (
        "Device Theft",
        "Potential exposure of sensitive organizational information stored on a stolen laptop"
    ),
    "RW-INJ-90": (
        "Suspected Phishing",
        "Potential malware infection or unauthorized access after employees opened a suspicious PDF"
    ),
    "RW-INJ-122": (
        "Suspected Data Exfiltration",
        "Significant data transferred to external IP addresses; possible exposure of sensitive information"
    ),
    "RW-INJ-124": (
        "Ransomware",
        "Computers locked by ransomware, disrupting operations and triggering a ransom demand"
    ),
    "RW-INJ-126": (
        "Data Breach",
        "Employee personal and financial information exposed; threatened sale of stolen data"
    ),
    "RW-INJ-128": (
        "Ransomware",
        "Media attention following the ransomware incident and data breach; reputational risk"
    ),

    # INSIDER THREAT: 12 events
    "IT-INJ-85": (
        "Vulnerability Exposure",
        "Potential unauthorized access to sensitive data through a microprocessor vulnerability"
    ),
    "IT-INJ-87": (
        "Vulnerability Exposure",
        "Costly and time-consuming hardware replacement; security risk remains during rollout"
    ),
    "IT-INJ-89": (
        "Accidental Data Disclosure",
        "Personally identifiable information accidentally disclosed to external contacts"
    ),
    "IT-INJ-91": (
        "No Specific Cyber Threat",
        "Workplace conflict following the PII disclosure; potential personnel consequences"
    ),
    "IT-INJ-93": (
        "Potential Insider Threat",
        "Risk of retaliation from a terminated employee; no cyberattack established at this stage"
    ),
    "IT-INJ-121": (
        "Vulnerability Exposure",
        "Extended security exposure and resource demands due to an 18-month replacement process"
    ),
    "IT-INJ-123": (
        "Potential Insider Threat",
        "Possible third-party access risk involving a former employee; no incident confirmed"
    ),
    "IT-INJ-125": (
        "Unauthorized File Modification",
        "Files moved or deleted; lost work since the last backup"
    ),
    "IT-INJ-127": (
        "Unauthorized Access",
        "Unauthorized administrator account discovered and deactivated; potential system compromise"
    ),
    "IT-INJ-129": (
        "Unauthorized Access",
        "Administrator credentials compromised, enabling privileged access and lateral movement"
    ),
    "IT-INJ-131": (
        "Suspicious Third-Party Activity",
        "Unauthorized account creation traced to a vendor-associated IP address; responsibility unconfirmed"
    ),
    "IT-INJ-133": (
        "Unauthorized File Deletion",
        "Critical organizational files appear deleted, threatening core business operations"
    ),

    # EXECUTIVES AND SENIOR LEADERSHIP: 2 events
    "EX-INJ-109": (
        "Whaling / Ransomware",
        "Payroll disruption, employee computer lockouts, ransom demand, and exposure of bank account information"
    ),
    "EX-INJ-126": (
        "Insider Threat",
        "Sensitive information stolen, including intellectual property; external data exposure and reputational risk"
    )
}

# Check that every event has a matching annotation
assert set(df_labeled["inject_id"]) == set(annotations)

# Assign threat types and business impacts
df_labeled["threat_type"] = df_labeled["inject_id"].map(lambda x: annotations[x][0])
df_labeled["business_impact"] = df_labeled["inject_id"].map(lambda x: annotations[x][1])

display(df_labeled[["inject_id", "manual_title", "threat_type", "business_impact"]])

,inject_id,manual_title,threat_type,business_impact
0,HC-INJ-87,Healthcare,Security Alert,No direct impact reported; increased awareness of threats targeting healthcare organizations
1,HC-INJ-89,Healthcare,Suspected Phishing,Potential credential theft and malware infection through suspicious emails and attachments
2,HC-INJ-117,Healthcare,No Specific Threat,No negative impact reported; patient monitor updates installed successfully
3,HC-INJ-119,Healthcare,Vulnerability Exposure,Potential compromise of clinical devices and systems if the vulnerability remains unpatched
4,HC-INJ-126,Healthcare,System Disruption (Unconfirmed Cause),MRI system delays causing patient appointments to run more than two hours behind schedule
5,HC-INJ-135,Healthcare,Unauthorized Access Risk,Potential misuse of former employees' active IT accounts; accounts were deactivated
6,HC-INJ-137,Healthcare,Data Integrity Issue,Incorrect patient records creating risks to patient safety and clinical decisions
7,HC-INJ-139,Healthcare,Medical Device Malfunction,Inaccurate monitoring data and incorrect infusion rates threatening patient safety
8,HC-INJ-156,Healthcare,Ransomware,"Files unavailable, $1 million ransom demand, and threatened exposure of patient data"
9,HC-INJ-159,Healthcare,Data Breach,"Patient information exposed; extortion attempts, complaints, and threatened legal action"


#### 5. Saving the results
The classified events are saved as a new CSV file for further analysis.

In [29]:
df_labeled.to_csv(
    data_dir / "cisa_classified_events.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Classified events saved successfully!")

Classified events saved successfully!
